# (Optional) Convert existing Delta files to Lance

> **This is an optional appendix, not part of the paved path.** The paved benchmark is `01a_delta_native` (Delta) vs `01b_lance_native` (Lance), both writing straight from generation. This notebook answers a narrower migration question for customers who are **already on Delta path-refs** and want the training-read performance Lance offers.

**Purpose:** Start from the state `01a_delta_native.ipynb` produces — **JPEG files already in a Volume** plus a **Delta table** referencing them — and convert it into a **Lance dataset with the image bytes stored inline**. The decision this models: the data has already landed on Delta; is it worth converting to Lance for training?

Because we begin from files-in-a-Volume, the Lance write here **includes reading every image back out of the Volume** — the same I/O the Delta path started from. That makes the write timing here directly comparable to `01a`'s Delta write: both move the image bytes through the Volume. (Contrast `01b_lance_native`, which never lands files at all — that is the true greenfield-ingest cost.)

| | Lance (this notebook) |
|---|---|
| Input | Delta table (`image_path` + metadata) + JPEG files in the Volume |
| Image storage | **Inline** JPEG bytes (blob layout) |
| Writer | Ray reads files → `write_fragments` + driver `commit` |
| Add a column | `add_columns` — new column only, no rewrite |

**Compute:** Databricks Classic Compute — same Ray/Spark split as `01a`.

---

**Inputs (from `01a_delta_native`, per size tier):**
- JPEG files at `/Volumes/{catalog}/{schema}/{volume}/synthetic_images_{size}/`
- Delta table `{catalog}.{schema}.synthetic_delta_{size}`

**Outputs (kept separate from the paved `01b` Lance dataset so it never clobbers it):**
- Lance dataset at `/Volumes/{catalog}/{schema}/{volume}/synthetic_lance_convert_{size}/`
- Metrics JSON at `/Volumes/{catalog}/{schema}/{volume}/artifacts/lance_convert_{size}.json`

**Next:** the migration cost compiles alongside the paved routes in `03_compile_results.ipynb`.

In [ ]:
# Must install before setup_ray_cluster — installing after restarts the Ray workers.
# ray[data]==2.54.1 pinned: 2.55.0+ added storage_options_provider to lance_datasink,
# a kwarg no released pylance version accepts.
# pyarrow floored not pinned — DBR preinstalls it; an exact pin risks a version conflict.
%pip install -qU "ray[data]==2.54.1" pylance numpy pandas "databricks-sdk>=0.49.0"
dbutils.library.restartPython()

In [ ]:
# ── Widgets ─────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("warehouse_id", "", "SQL Warehouse ID (blank = provision)")
dbutils.widgets.text("seed", "42", "RNG seed")
dbutils.widgets.text("embedding_dim", "512", "Embedding dim")

size          = dbutils.widgets.get("size")
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
volume        = dbutils.widgets.get("volume")
SEED          = int(dbutils.widgets.get("seed"))
EMBEDDING_DIM = int(dbutils.widgets.get("embedding_dim"))

SIZE_MAP = {"10k": 10_000, "100k": 100_000, "1m": 1_000_000, "10m": 10_000_000}
N_ROWS   = SIZE_MAP[size]

# Fixed category set — MUST match 01a_delta_native + 01b_lance_native.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]

base_vol      = f"/Volumes/{catalog}/{schema}/{volume}"
lance_subdir  = f"synthetic_lance_convert_{size}"             # output: kept separate from 01b's paved synthetic_lance_{size}
lance_path    = f"{base_vol}/{lance_subdir}"                  # output: Lance dataset (inline bytes)
images_dir    = f"{base_vol}/synthetic_images_{size}"         # input: JPEG files written by 01a_delta_native
delta_table   = f"{catalog}.{schema}.synthetic_delta_{size}"  # input: metadata + image_path
artifacts_dir = f"{base_vol}/artifacts"                       # write metrics here for notebook 03

print(f"Size tier   : {size} ({N_ROWS:,} rows)")
print(f"Lance out   : {lance_path}")
print(f"Delta in    : {delta_table}")
print(f"Images in   : {images_dir}")
print(f"Artifacts   : {artifacts_dir}")


In [ ]:
import os

# Credentials — set BEFORE setup_ray_cluster so Ray workers inherit them (Ray 2.41+).
os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [ ]:
# Ensure Ray tmp + artifacts dirs exist. Images and the Delta table are
# produced by 01a_delta_native — verify the image dir is present before reading.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
ray_tmp_path = f"/Volumes/{catalog}/{schema}/ray_tmp"
os.makedirs(artifacts_dir, exist_ok=True)
assert os.path.isdir(images_dir), (
    f"{images_dir} not found — run 01a_delta_native for size={size} first."
)
print(f"Ray tmp     : {ray_tmp_path}")
print(f"Images dir  : {images_dir}  (input)")
print(f"Artifacts   : {artifacts_dir}")


## SQL Warehouse — provision or reuse

`ray.data.write_databricks_table` routes through a running SQL Warehouse. Provision-or-reuse by name so reruns don't spawn duplicates; serverless + short auto-stop keeps an idle warehouse from billing. Pin an existing one via the `warehouse_id` widget.

In [ ]:
from databricks.sdk.service.sql import State

WAREHOUSE_NAME = "ray-benchmark-warehouse"


def get_or_create_warehouse(warehouse_id="", name=WAREHOUSE_NAME,
                            cluster_size="Small", auto_stop_mins=10):
    if warehouse_id:
        return warehouse_id
    for wh in w.warehouses.list():
        if wh.name == name:
            if wh.state in (State.STOPPED, State.STOPPING):
                w.warehouses.start(wh.id).result()
            elif wh.state == State.STARTING:
                w.warehouses.get_and_wait(wh.id)
            print(f"Reusing warehouse '{name}' ({wh.id})")
            return wh.id
    created = w.warehouses.create(
        name=name, cluster_size=cluster_size, auto_stop_mins=auto_stop_mins,
        enable_serverless_compute=True, min_num_clusters=1, max_num_clusters=1,
    ).result()
    print(f"Created warehouse '{name}' ({created.id})")
    return created.id


warehouse_id = get_or_create_warehouse(dbutils.widgets.get("warehouse_id"))

In [ ]:
# Classic Compute Ray cluster.
# N_WORKER_NODES allocated to Ray; remaining nodes stay available for Spark
# (write_databricks_table, DESCRIBE DETAIL, ALTER TABLE, etc.).
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

N_WORKER_NODES = 6
CPUS_PER_NODE  = 16

setup_ray_cluster(
    min_worker_nodes=N_WORKER_NODES,
    max_worker_nodes=N_WORKER_NODES,      # fixed size (min == max)
    num_cpus_worker_node=CPUS_PER_NODE,
    num_gpus_worker_node=0,
    collect_log_to_path=ray_tmp_path,
)
ray.init(address="auto", ignore_reinit_error=True)

total_cpus = ray.cluster_resources().get("CPU", 0)
print(f"Total CPUs  : {total_cpus:.0f} | alive nodes: {sum(1 for n in ray.nodes() if n['Alive'])}")
assert total_cpus >= N_WORKER_NODES * CPUS_PER_NODE * 0.9, "Cluster did not fully start"

## Read images from the Volume (via Ray)

Read the metadata + `image_path` from the Delta table through the SQL Warehouse, then issue a per-image Volume GET to pull each JPEG’s bytes back in. The result is an in-memory Ray dataset carrying the full schema with inline `image` bytes — exactly what the Lance write consumes. This read is the cost Lance conversion pays that a Delta path-ref table does not.

In [ ]:
import time
import numpy as np

os.environ["RAY_UC_VOLUMES_FUSE_TEMP_DIR"] = ray_tmp_path


def read_image_from_path(batch):
    """Fetch JPEG bytes from each image_path (per-image Volume GET); keep metadata."""
    imgs = []
    for p in batch["image_path"]:
        with open(p, "rb") as f:
            imgs.append(f.read())
    return {
        "id":         batch["id"],
        "image":      np.asarray(imgs, dtype=object),
        "caption":    batch["caption"],
        "embedding":  batch["embedding"],
        "category":   batch["category"],
        "brightness": batch["brightness"],
        "quality":    batch["quality"],
    }


t0 = time.time()
ds = ray.data.read_databricks_tables(
    warehouse_id=warehouse_id, catalog=catalog, schema=schema,
    query=(
        f"SELECT id, image_path, caption, embedding, category, brightness, quality "
        f"FROM {delta_table}"
    ),
).map_batches(read_image_from_path, batch_size=512).materialize()
read_s = time.time() - t0
print(f"Read {ds.count():,} rows (images inlined) in {read_s:6.2f}s")

total_image_bytes = ds.map_batches(
    lambda b: {"nbytes": np.array([sum(len(x) for x in b["image"])])},
    batch_size=512,
).sum("nbytes")
print(f"Raw image bytes: {total_image_bytes / 1e9:.3f} GB")

## Write — Lance (inline)

Each Ray write task emits an independent Lance fragment; a single driver-side
commit merges fragment metadata into a new dataset version. Image bytes stored inline.

The next cell uses a manual `write_fragments` + driver-side `commit` — not a one-call
high-level writer — for one Databricks-specific reason: Lance's default commit path
finalises a write with a POSIX `rename()`, which the UC Volume FUSE mount does not
implement (`ENOSYS`). Writing to the Volume's underlying `s3://` URI instead commits
with an S3-native atomic `PutObject`; the files land in the same Volume and are readable
through `/Volumes/...` immediately after. Splitting the write into distributed
`write_fragments` + a single explicit `commit` is what lets each Ray task write straight
to that `s3://` URI in parallel, with only fragment metadata returning to the driver.

In [ ]:
import base64, os, pickle, time, lance
import boto3 as _boto3
import pyarrow as pa
import numpy as np
from lance.fragment import write_fragments


# Derive the S3 URI that backs this managed Volume.
# Lance's local-filesystem commit path uses POSIX rename(), which the UC FUSE driver
# does not implement. Writing directly to the S3 URI uses S3-native atomic PutObject
# instead, bypassing FUSE. Files land in the same managed Volume and are readable
# via /Volumes/... immediately after commit.
vol_info     = w.volumes.read(f"{catalog}.{schema}.{volume}")
lance_s3_path = f"{vol_info.storage_location.rstrip('/')}/{lance_subdir}"
# Captured in worker closures — avoids a second SDK call inside each task.
_vol_id = vol_info.volume_id
_aws_region = _boto3.session.Session().region_name or "us-west-2"


def dir_stats(path):
    total, nfiles = 0, 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f)); nfiles += 1
            except OSError:
                pass
    return total, nfiles


def _s3_storage_options() -> dict:
    """UC credential vending — get temporary S3 credentials for the managed volume.

    Databricks workers access managed storage through UC credential vending, not raw
    EC2 instance-profile IAM (boto3 returns None because the profile isn't wired into
    the Ray worker process). DATABRICKS_HOST + DATABRICKS_TOKEN are set in env by the credentials cell
    (before setup_ray_cluster so all workers inherit them).
    """
    import requests, os
    resp = requests.post(
        f"{os.environ['DATABRICKS_HOST']}/api/2.1/unity-catalog/temporary-volume-credentials",
        headers={"Authorization": f"Bearer {os.environ['DATABRICKS_TOKEN']}"},
        json={"volume_id": _vol_id, "operation": "WRITE_VOLUME"},
        timeout=10,
    )
    if not resp.ok:
        raise RuntimeError(f"UC credential vending {resp.status_code}: {resp.text}")
    aws = resp.json()["aws_temp_credentials"]
    return {
        "aws_access_key_id":     aws["access_key_id"],
        "aws_secret_access_key": aws["secret_access_key"],
        "aws_session_token":     aws.get("session_token", ""),
        "aws_region":            _aws_region,
    }


def _write_frags(batch: pa.Table) -> dict:
    """Distributed Lance fragment write via write_fragments; driver-side commit later.

    Normalize Ray's Arrow blocks to plain Arrow types Lance accepts:
    * image     -> large_binary
    * embedding -> list<float>
    * strings   -> large_string
    """
    cols = batch.to_pydict()
    normalized = pa.Table.from_pydict(
        {
            "id": pa.array(cols["id"], type=pa.int64()),
            "image": pa.array([bytes(x) for x in cols["image"]], type=pa.large_binary()),
            "caption": pa.array(cols["caption"], type=pa.large_string()),
            "embedding": pa.array(
                [np.asarray(x, dtype=np.float32).tolist() for x in cols["embedding"]],
                type=pa.list_(pa.float32()),
            ),
            "category": pa.array(cols["category"], type=pa.large_string()),
            "brightness": pa.array(cols["brightness"], type=pa.float32()),
            "quality": pa.array(cols["quality"], type=pa.int32()),
        }
    )
    schema = normalized.schema
    fragments = write_fragments(
        normalized.to_reader(), lance_s3_path, schema=schema,
        storage_options=_s3_storage_options(),
    )
    return {
        "fragment_b64": np.asarray([
            base64.b64encode(pickle.dumps(fragment)).decode("ascii") for fragment in fragments
        ], dtype=object),
        "schema_b64": np.asarray([
            base64.b64encode(pickle.dumps(schema)).decode("ascii") for _ in fragments
        ], dtype=object),
    }


t0 = time.time()
fragment_rows = ds.map_batches(_write_frags, batch_format="pyarrow").take_all()
fragments, lance_schema = [], None
for row in fragment_rows:
    fragments.append(pickle.loads(base64.b64decode(row["fragment_b64"])))
    lance_schema = pickle.loads(base64.b64decode(row["schema_b64"]))

op = lance.LanceOperation.Overwrite(lance_schema, fragments)
lance.LanceDataset.commit(lance_s3_path, op, storage_options=_s3_storage_options())
lance_write_s = time.time() - t0

lds = lance.dataset(lance_path)
n_frag = len(lds.get_fragments())
lc_bytes, _ = dir_stats(lance_path)
rows_per_s = N_ROWS / lance_write_s
mb_per_s = (lc_bytes / 1e6) / lance_write_s
print(f"Lance write   : {lance_write_s:6.2f}s | {rows_per_s:>10,.0f} rows/s | {mb_per_s:6.1f} MB/s")
print(f"Lance on-disk : {lc_bytes / 1e9:.3f} GB across {n_frag} fragments")

## Verify — round-trip + random access

Confirm the Delta path-referenced JPEG round-trips the same bytes Lance stored inline for the same `id`, and time Lance point lookups at start / middle / end — access cost should be roughly constant (O(1) fragment addressing), independent of row position.

In [ ]:
probe_ids = [0, N_ROWS // 2, N_ROWS - 1]

# Lance take([n]) addresses by ROW INDEX within the dataset, not by the id column.
# Use to_table(filter=...) to look up rows by id value for a correct comparison.
# The raw-take timing below still shows the O(1) fragment-address cost.
print("Lance random-access latency (row-index addressing):")
for idx in probe_ids:
    t0 = time.time()
    lds.take([idx])   # timings only — row at this index may have any id
    print(f"  take index={idx:>12,}: {(time.time() - t0) * 1000:6.2f} ms")

print("\nLance id-value lookup (for round-trip comparison):")
lance_rows = {}
for pid in probe_ids:
    t0 = time.time()
    tbl = lds.to_table(filter=f"id = {pid}", columns=["id", "image"])
    row = tbl.to_pylist()[0]
    lance_rows[pid] = row["image"]
    print(f"  scan id={pid:>12,}: {(time.time() - t0) * 1000:6.2f} ms")

# Delta round-trip: read image_path, then GET the file (the per-image hop the benchmark measures)
pdf = spark.sql(
    f"SELECT id, image_path FROM {delta_table} WHERE id IN ({','.join(map(str, probe_ids))})"
).toPandas()
path_map = dict(zip(pdf["id"], pdf["image_path"]))

roundtrip_ok = True
print("\nRound-trip (Delta file == Lance inline):")
for pid in probe_ids:
    with open(path_map[pid], "rb") as f:
        delta_bytes = f.read()
    ok = delta_bytes == lance_rows.get(pid)
    roundtrip_ok = roundtrip_ok and ok
    print(f"  id={pid:>12,}: {'OK' if ok else 'MISMATCH':>8}  ({len(delta_bytes) / 1024:.0f} KB)")
print(f"\nround-trip all OK: {roundtrip_ok}")

## ETL benchmark — backfill a new column

Compute a derived column once and add it to the existing dataset. Lance's `add_columns` writes only the new column; Delta must `ALTER TABLE ADD COLUMN` then backfill (rewrites the affected Parquet files). Derived column: the L2 norm of the embedding — a stand-in for any UDF-computed feature.

In [ ]:
import pyarrow as pa


def compute_norm(record_batch):
    """BatchUDF: receives a pyarrow.RecordBatch, returns the new column."""
    embs = np.stack(record_batch.column("embedding").to_pylist()).astype("float32")
    norms = np.linalg.norm(embs, axis=1).astype("float32")
    return pa.record_batch({"embedding_norm": pa.array(norms)})


# ── Lance: add_columns — no rewrite of existing data ───────────────────────
# add_columns commits a new manifest version — same FUSE rename wall as the
# original write. Open from lance_s3_path + credentials so the commit goes
# through S3-native PutObject instead of POSIX rename.
lds_rw = lance.dataset(lance_s3_path, storage_options=_s3_storage_options())
t0 = time.time()
lds_rw.add_columns(compute_norm, read_columns=["embedding"])
lance_backfill_s = time.time() - t0
lc_bytes_after, _ = dir_stats(lance_path)
print(f"Lance add_columns : {lance_backfill_s:6.2f}s | +{(lc_bytes_after - lc_bytes) / 1e6:,.1f} MB (new column only)")

In [ ]:
import json

# ── common block ── identical key names across 01a / 01b (and this optional 01) so 03
# can stack the artifacts into one table with no per-format key mapping. Format-specific
# detail is kept below in `raw`.
common = {
    "path_label":        "lance_convert",                   # optional 01 = convert existing files to Lance
    "write_total_s":     round(read_s + lance_write_s, 3),  # end-to-end: readback + Lance write
    "target_write_s":    round(lance_write_s, 3),           # Lance write only (readback excluded)
    "n_output_files":    int(n_frag),                       # fragments — vs 01a's per-image files
    "on_disk_bytes":     int(lc_bytes),
    "etl_backfill_s":    round(lance_backfill_s, 3),
    "etl_bytes_written": int(lc_bytes_after - lc_bytes),    # new-column-only write
    "roundtrip_ok":      bool(roundtrip_ok),
}

lance_metrics = {
    "size":   size,
    "n_rows": int(N_ROWS),
    "common": common,
    "raw": {                                                # format-specific detail
        "read_s":           round(read_s, 3),
        "lance_write_s":    round(lance_write_s, 3),
        "lc_bytes":         int(lc_bytes),
        "n_frag":           int(n_frag),
        "lance_backfill_s": round(lance_backfill_s, 3),
        "lc_bytes_after":   int(lc_bytes_after),
        "roundtrip_ok":     bool(roundtrip_ok),
    },
}
out_path = f"{artifacts_dir}/lance_convert_{size}.json"
with open(out_path, "w") as f:
    json.dump(lance_metrics, f, indent=2)
print(f"Wrote {out_path}")
print(json.dumps(lance_metrics, indent=2))

## Done — migration (convert) artifact ready

The inline Lance dataset is written to `synthetic_lance_convert_{size}` (separate from the paved `01b` dataset) and round-trip-verified against the Delta files. Metrics are in `artifacts/lance_convert_{size}.json`.

**Next:** `03_compile_results.ipynb` folds this migration cost in alongside the paved `01a` (Delta) vs `01b` (Lance-native) comparison — the extra `read_s` here is the price of converting an existing file dataset rather than writing Lance from the source.